# Parameters

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd

# ===== CONFIGURAÇÃO DE CAMINHOS =====
current_notebook = Path.cwd()  
project_root = current_notebook.parent.parent 

# Adiciona o diretório raiz ao sys.path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Adiciona o diretório Modules ao sys.path
modules_dir = project_root / "Modules"
if str(modules_dir) not in sys.path:
    sys.path.insert(0, str(modules_dir))

# ===== IMPORTS DOS MÓDULOS =====
import Modules.ClusterGmmModule as cluster
import Modules.FutureAnalysisModule as fa
from Modules.config import CONFIG

# ===== CONFIGURAÇÕES DO PROJETO =====
DATAPATH = CONFIG["datapath"]
COVID_TRAIN_DATA_FILE = CONFIG["covid_train_data_file"]
COVID_TEST_DATA_FILE = CONFIG["covid_test_data_file"]
FUTURE_DATA_FILE = CONFIG["future_data_file"]

CONTROL_GROUP_TRAIN = CONFIG["control_group_train"]
CONTROL_GROUP_TEST = CONFIG["control_group_test"]
CONTROL_GROUP_READMISSION = CONFIG["control_group_readmission"]

FIGSIZE_CLUSTER_HEATMAP = CONFIG["figsize_cluster_heatmap"]
FIGSIZE_FUTURE_HEATMAP = CONFIG["figsize_future_heatmap"]
IMAGES_SAVE_PATH = CONFIG["image_save_path"]
TRIALS_OPTUNA = 80

# Import data

In [ ]:
# ===== CARREGAMENTO DOS DADOS =====
data_folder = current_notebook / DATAPATH

covid_train = pd.read_csv(data_folder / COVID_TRAIN_DATA_FILE)
covid_test = pd.read_csv(data_folder / COVID_TEST_DATA_FILE)
future_data = pd.read_csv(data_folder / FUTURE_DATA_FILE)

control_train = pd.read_csv(data_folder / CONTROL_GROUP_TRAIN)
control_test = pd.read_csv(data_folder / CONTROL_GROUP_TEST)
control = pd.concat([control_train, control_test], axis=0)
control_readmission = pd.read_csv(data_folder / CONTROL_GROUP_READMISSION)

# Feature engineering: morte após a internação
covid_train['died_after'] = ((covid_train['died'] == 1) & (covid_train['died_in_stay'] == 0)).astype(int)
covid_test['died_after'] = ((covid_test['died'] == 1) & (covid_test['died_in_stay'] == 0)).astype(int)
future_data['died_after'] = ((future_data['died'] == 1) & (future_data['died_in_stay'] == 0)).astype(int)

In [ ]:
data_covid = pd.concat([covid_train, covid_test], axis=0)
data_covid = data_covid.sample(frac=1, random_state=42).reset_index(drop=True)

# Mice Data

In [ ]:
categorical_features = [
            "myocardial_infarct",
            "congestive_heart_failure",
            "peripheral_vascular_disease",
            "cerebrovascular_disease",
            "dementia",
            "chronic_pulmonary_disease",
            "rheumatic_disease",
            "peptic_ulcer_disease",
            "mild_liver_disease",
            "diabetes_without_cc",
            "diabetes_with_cc",
            "paraplegia",
            "renal_disease",
            "malignant_cancer",
            "severe_liver_disease",
            "metastatic_solid_tumor",
            "aids",
            "gender_M",
            "died_in_stay",
            "died_after",
            "died",
            "COVID"
        ]

In [ ]:
features_not_considered = ['died', 'died_in_stay', 'died_after', 'COVID', 'subject_id', 'hadm_id']

In [ ]:
helper = cluster.GmmClusterHelper(data=data_covid, features_not_considered=features_not_considered)

## Find best hyperparameters for GMM

In [ ]:
param = {
    "n_components": {"min": 2, "max": 15},
    "covariance_type": ["full", "tied", "diag", "spherical"]
}

### DBCV

In [ ]:
os.environ["PYTHONWARNINGS"] = "ignore"
pca_dbcv_df, pca_dbcv_param, pca_dbcv_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True,
    suffix="pca",
    metric="dbcv",
    dimensionality_reduction = {'method': 'PCA', 'dimensions': 30}
)

In [ ]:
pca_dbcv_param = {'n_components': 11, 'covariance_type': 'full'}

In [ ]:
os.environ["PYTHONWARNINGS"] = "ignore"
ae_dbcv_df, ae_dbcv_param, ae_dbcv_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True,
    suffix="ae",
    metric="dbcv",
    dimensionality_reduction = {'method': 'AE', 'dimensions': 10}
)

In [ ]:
ae_dbcv_param = {'n_components': 12, 'covariance_type': 'diag'}

### DISCO

In [ ]:
os.environ["PYTHONWARNINGS"] = "ignore"
pca_disco_df, pca_disco_param, pca_disco_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True,
    suffix="pca",
    metric="disco",
    dimensionality_reduction = {'method': 'PCA', 'dimensions': 30}
)

In [ ]:
pca_disco_param = {'n_components': 2, 'covariance_type': 'diag'}

In [ ]:
os.environ["PYTHONWARNINGS"] = "ignore"
ae_disco_df, ae_disco_param, ae_disco_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True,
    suffix="ae",
    metric="disco",
    dimensionality_reduction = {'method': 'AE', 'dimensions': 10}
)

In [ ]:
ae_disco_param = {'n_components': 2, 'covariance_type': 'diag'}

### DSI

In [ ]:
os.environ["PYTHONWARNINGS"] = "ignore"
pca_dsi_df, pca_dsi_param, pca_dsi_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True,
    suffix="pca",
    metric="dsi",
    dimensionality_reduction = {'method': 'PCA', 'dimensions': 30}
)

In [ ]:
pca_dsi_param = {'n_components': 2, 'covariance_type': 'spherical'}

In [ ]:
os.environ["PYTHONWARNINGS"] = "ignore"
ae_dsi_df, ae_dsi_param, ae_dsi_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True,
    suffix="ae",
    metric="dsi",
    dimensionality_reduction = {'method': 'AE', 'dimensions': 10}
)

In [ ]:
ae_dsi_param = {'n_components': 2, 'covariance_type': 'diag'}

### Silhouette

In [ ]:
os.environ['PYTHONWARNINGS'] = 'ignore'
pca_silhouette_df, pca_silhouette_param, pca_silhouette_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True,
    suffix="pca",
    metric="silhouette",
    dimensionality_reduction = {'method': 'PCA', 'dimensions': 30}
)

In [ ]:
pca_silhouette_param = {'n_components': 2, 'covariance_type': 'spherical'}

In [ ]:
os.environ['PYTHONWARNINGS'] = 'ignore'
ae_silhouette_df, ae_silhouette_param, ae_silhouette_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True,
    suffix="ae",
    metric="silhouette",
    dimensionality_reduction = {'method': 'AE', 'dimensions': 10}
)

In [ ]:
ae_silhouette_param = {'n_components': 2, 'covariance_type': 'diag'}

### Metrics

In [ ]:
helper.clustering(
    n_components=pca_dbcv_param["n_components"],
    covariance_type=pca_dbcv_param["covariance_type"],
    dimensionality_reduction = {'method': 'PCA', 'dimensions': 30}
)
helper.get_metrics()

In [ ]:
helper.clustering(
    n_components=pca_disco_param["n_components"],
    covariance_type=pca_disco_param["covariance_type"],
    dimensionality_reduction={"method": "PCA", "dimensions": 30},
)
helper.get_metrics()

In [ ]:
helper.clustering(
    n_components=pca_dsi_param["n_components"],
    covariance_type=pca_dsi_param["covariance_type"],
    dimensionality_reduction={"method": "PCA", "dimensions": 30},
)
helper.get_metrics()

In [ ]:
helper.clustering(
    n_components=pca_silhouette_param["n_components"],
    covariance_type=pca_silhouette_param["covariance_type"],
    dimensionality_reduction={"method": "PCA", "dimensions": 30},
)
helper.get_metrics()

In [ ]:
helper.clustering(
    n_components=ae_dbcv_param["n_components"],
    covariance_type=ae_dbcv_param["covariance_type"],
    dimensionality_reduction={"method": "AE", "dimensions": 10},
)
helper.get_metrics()

In [ ]:
helper.clustering(
    n_components=ae_disco_param["n_components"],
    covariance_type=ae_disco_param["covariance_type"],
    dimensionality_reduction={"method": "AE", "dimensions": 10},
)
helper.get_metrics()

In [ ]:
helper.clustering(
    n_components=ae_dsi_param["n_components"],
    covariance_type=ae_dsi_param["covariance_type"],
    dimensionality_reduction={"method": "AE", "dimensions": 10},
)
helper.get_metrics()

In [ ]:
helper.clustering(
    n_components=ae_silhouette_param["n_components"],
    covariance_type=ae_silhouette_param["covariance_type"],
    dimensionality_reduction={"method": "AE", "dimensions": 10},
)
helper.get_metrics()

## Best

In [ ]:
best_param, best_dr = (pca_silhouette_param, {'method': 'AE', 'dimensions': 10})

In [ ]:
helper.clustering(
    n_components=best_param["n_components"],
    covariance_type=best_param["covariance_type"],
    dimensionality_reduction=best_dr,
)
helper.heatmap_clusters_categorical(
    figsize=FIGSIZE_CLUSTER_HEATMAP,
    savepath=IMAGES_SAVE_PATH + "gmm-dr-categorical",
)

In [ ]:
selected_clusters = [0, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12]

In [ ]:
helper.show_cluster_compare_numerical(
    selected_clusters=selected_clusters,
    figsize=(10, 40),
    savepath=IMAGES_SAVE_PATH + "gmm-dr-numerical", verbose=1)

In [ ]:
helper.set_clustered_autoencoder()

In [ ]:
helper.show_clustered_autoencoder(selected_clusters=selected_clusters, savepath=IMAGES_SAVE_PATH + "gmm-autoencoder-dr")

### Future Data

In [ ]:
future_helper = fa.FutureAnalysisHelper(
    helper.clustered_data, future_data, control, control_readmission
)
delta = future_helper.get_delta_clusters(percentage=True, relative_total=True)
future_helper.show_delta_heatmap(
    figsize=FIGSIZE_FUTURE_HEATMAP,
    relative_total=True,
    selected_clusters=selected_clusters,
    savepath=IMAGES_SAVE_PATH + "gmm-dr-future",
)

In [ ]:
future_helper.get_mean_readmission()

In [ ]:
future_helper.get_mean_days_gap()

In [ ]:
future_helper.get_mortality_rates(only_first_admission=True)

# Add Log

In [ ]:
metrics = helper.get_metrics()

log_file = "../metrics.csv"
current_dir = os.getcwd()
log_file_path = os.path.join(current_dir, log_file)

# Add line to save log
if os.path.exists(log_file_path):
    with open(log_file_path, 'a') as f:
        f.write(f"GMM, Autoencoder, Comprehensive, {metrics['disco']}, {metrics['dbcv']}, {metrics['dsi']}, {metrics['silhouette']}\n")